# Assignment 4: Futures Settlement and IFR Yield Curves

**Aditya Amit Shah | as22008**  
FRE-GY 9743, Fall 2026

The written answer is followed by the completed, reproducible computational assignment. Rates are annual decimals throughout.


## Part I. Why discount OTC cash flows when futures settle every day?

**The daily transfers have different economic meanings.** For a conventional collateralized OTC trade, the payment is security for an obligation that still settles in the future. For a standard futures contract, variation margin settles that day's profit or loss and resets the continuing contract's value to zero.

### OTC: collateral secures the claim and carries interest

Suppose an OTC trade is worth $10 to me today and my counterparty posts $10 of cash collateral. I now hold $10 of protection, but the original trade still has a $10 replacement value. The collateral must be returned or applied against the eventual payment, and I generally owe the poster the agreed collateral interest. Receiving it does not give me an extra $10 of unconditional profit. A zero **uncollateralized exposure** is therefore different from a zero **trade value**.

The future contractual payment still needs to be converted into today's money. Under ideal full cash collateralization (one currency, no thresholds, no default or funding frictions), its discount rate is the collateral remuneration rate $c$. For a payment $X_T$,

$$V_t = \mathbb{E}_t^{Q^c}\!\left[D_c(t,T)X_T\right],\qquad
D_c(t,T)=\exp\!\left(-\int_t^T c_s\,ds\right),$$

where $Q^c$ denotes the pricing measure consistent with those collateralized funding assumptions. The usual risk-neutral measure is recovered when the collateral rate is the risk-free money-market rate.

The interest also explains why "MTM minus existing collateral" does not make an OTC trade identical to a future. If the initial collateral balance is $V_k$ and interest is capitalized at $c_k$ over $\Delta t$, the next top-up is

$$V_{k+1}-(1+c_k\Delta t)V_k.$$

If collateral interest is paid separately instead, the net cash flows give the same economics. Daily collateralization changes the credit exposure and the appropriate discount rate; it does not remove time value.

### Futures: daily variation margin is realized profit or loss

If my long future rises from 100 to 110, I receive 10 as settlement of that gain. Immediately after settlement, the ongoing futures contract has zero value. If I close the position at that price, I retain the settled profit; it is not returnable collateral for an unpaid terminal claim. Initial margin is separate and is omitted from this frictionless pricing argument.

Let $F_k=F(t_k,T)$. With the next period's discount factor $D_k$ known at $t_k$, as the assignment assumes,

$$0=\mathbb{E}_k^Q[D_k(F_{k+1}-F_k)]
=D_k\left(\mathbb{E}_k^Q[F_{k+1}]-F_k\right).$$

Thus $F_k=\mathbb{E}_k^Q[F_{k+1}]$ and, by repeated conditioning,

$$\boxed{F(t_k,T)=\mathbb{E}_k^Q[S_T].}$$

**Discounting was applied and then cancelled from a zero-value equation.** This is an equation for the quoted futures price, not a claim that future money has no time value. An ordinary terminal payment still requires discounting, and rates can still affect futures through the underlying's carry and the investment or funding of daily settlements.

### Scope of the distinction

The cash-flow terms matter more than the trading venue. Some cleared OTC swaps also settle to market and include price alignment interest or adjustments that preserve collateralized-swap economics. Thus "OTC versus exchange" is shorthand here for the conventional collateralized OTC trade versus the standard daily-settled future described in the question.

Under a common risk-free/collateral rate assumption, a forward's fair delivery price is

$$K_t=\frac{\mathbb{E}_t^Q[D(t,T)S_T]}{\mathbb{E}_t^Q[D(t,T)]}.$$

It equals the futures quote when discounting to maturity is deterministic. With stochastic rates it generally differs because the discount factor and the terminal asset price can be correlated. The assignment's futures argument allows future rates to change, but requires each next settlement period's rate to be known at its start. If even that next-period discount factor is random, it cannot be pulled out of the conditional expectation.

**Sources:** [BIS, settlement-to-market versus collateralized-to-market](https://www.bis.org/media-releases/20180920-frequently-asked-questions-liquidity-risk-treatment-settled-market-derivatives); [Lou, *Derivatives Discounting Explained*, Section 2.1](https://arxiv.org/pdf/2002.08532); [CME, *Bank Asset/Liability Management: Swaps and Swap Futures*, pp. 2-3](https://www.cmegroup.com/education/files/bank-asset-liability-management-swaps-and-swap-futures.pdf).

## Part II. Computational tasks

The starter's dates and IFR inputs are retained. The model combines a SOFR curve and a funding-spread component. Run all cells in order from the repository root.


In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

from fixedincomelib import (
    qfAddPeriod,
    qfAccrued,
    qfCreateData1D,
    qfCreateDataCollection,
    qfCreateBuildMethod,
    qfCreateModelBuildMethodCollection,
    qfCreateModel,
    qfDiscountFactor,
)
from fixedincomelib.date import Date

## Library implementation

All six library TODOs are complete: the four piecewise-constant interpolator methods, the component discount factor, and recursive composition of referenced curves. The notebook completes product creation, pricing, and analytic risk below.


## 1. Input data

Rates are annual decimals: `0.03 = 3%`. Edit the inputs here and rerun the following cells. The funding component adds an IFR spread over SOFR.

In [2]:
value_date = "2026-10-07"
payment_currency = "USD"  # Placeholder; currently ignored by the model.
query_tenors = ["6M", "1Y", "2Y", "5Y", "10Y", "15Y"]

projection_ifr = pd.DataFrame(
    {
        "values": [
            0.030, 0.031, 0.032, 0.033, 0.034,
            0.035, 0.036, 0.037, 0.038, 0.039,
        ]
    },
    index=pd.Index([f"{year}Y" for year in range(1, 11)], name="Tenor"),
)

funding_ifr = pd.DataFrame(
    {"values": [0.0]},
    index=pd.Index(["1Y"], name="Tenor"),
)

display(projection_ifr.rename(columns={"values": "SOFR-1B IFR"}))
display(funding_ifr.rename(columns={"values": "SOFR-1B-FLAT IFR spread"}))

,SOFR-1B IFR
Tenor,
1Y,0.030
2Y,0.031
3Y,0.032
4Y,0.033
5Y,0.034
6Y,0.035
7Y,0.036
8Y,0.037
9Y,0.038


,SOFR-1B-FLAT IFR spread
Tenor,
1Y,0.0


## 2. Data and build methods

The registered IFR conventions identify each dataset. Each build method maps its dataset to a target component.

In [3]:
projection_data = qfCreateData1D(
    "INSTANTANEOUS FORWARD RATE",
    "SOFR-1B-IFR",
    projection_ifr,
)

funding_data = qfCreateData1D(
    "INSTANTANEOUS FORWARD RATE",
    "SOFR-1B-FUNDING-IFR",
    funding_ifr,
)

data_collection = qfCreateDataCollection([projection_data, funding_data])
display(data_collection.display())

,Data Shape,Data Type,Data Convention
0,DATA1D,INSTANTANEOUS FORWARD RATE,SOFR-1B-IFR
1,DATA1D,INSTANTANEOUS FORWARD RATE,SOFR-1B-FUNDING-IFR


In [4]:
projection_bm = qfCreateBuildMethod(
    "YIELD_CURVE_INDEX",
    {
        "TARGET": "SOFR-1B",
        "INSTANTANEOUS FORWARD RATE": "SOFR-1B-IFR",
    },
)

funding_bm = qfCreateBuildMethod(
    "YIELD_CURVE_FUNDING",
    {
        "TARGET": "SOFR-1B-FLAT",
        "REFERENCE INDEX":"SOFR-1B",
        "INSTANTANEOUS FORWARD RATE": "SOFR-1B-FUNDING-IFR",
    },
)

build_methods = qfCreateModelBuildMethodCollection([projection_bm, funding_bm])

## 3. Build the model

Trivial calibration copies IFR values into state. Tenors use Following/USGS; curve times use ACT/ACT (ISDA). Defaults are piecewise-constant left-continuous interpolation and flat extrapolation.

The registered reference is `SOFR-1B-FLAT → SOFR-1B`.

In [5]:
model = qfCreateModel(
    value_date,
    "YIELD_CURVE",
    data_collection,
    build_methods,
)

projection_component = model.retrieve_model_component(projection_bm.target_index)
funding_component = model.retrieve_model_component(funding_bm.target_index)

state_table = pd.concat(
    [
        pd.DataFrame({
            "Component": projection_bm.target,
            "Tenor": projection_data.axis1,
            "Time (years)": projection_component.state_data[0],
            "IFR": projection_component.state_data[1],
        }),
        pd.DataFrame({
            "Component": funding_bm.target,
            "Tenor": funding_data.axis1,
            "Time (years)": funding_component.state_data[0],
            "IFR": funding_component.state_data[1],
        }),
    ],
    ignore_index=True,
)

display(state_table)

,Component,Tenor,Time (years),IFR
0,SOFR-1B,1Y,1.000000,0.030
1,SOFR-1B,2Y,2.008840,0.031
2,SOFR-1B,3Y,3.005479,0.032
3,SOFR-1B,4Y,4.000000,0.033
4,SOFR-1B,5Y,5.000000,0.034
5,SOFR-1B,6Y,6.000644,0.035
6,SOFR-1B,7Y,7.000000,0.036
7,SOFR-1B,8Y,8.008219,0.037
8,SOFR-1B,9Y,9.005479,0.038
9,SOFR-1B,10Y,10.000644,0.039


## 4. Discount factors

Each component returns $D_{\mathrm{local}}(t)=\exp(-\int_0^t f(s)\,ds)$. The model follows references and multiplies their factors:

$$D_{\mathrm{funding}}(t)=D_{\mathrm{SOFR}}(t)\,D_{\mathrm{funding,\ local}}(t).$$

With zero funding IFR, the local factor is 1 and the two complete curves agree.

In [6]:
query_dates = [value_date] + [
    qfAddPeriod(value_date, tenor, "F", "USGS")
    for tenor in query_tenors
]

discount_factors = pd.DataFrame({
    "Tenor": ["Value date"] + query_tenors,
    "Date": query_dates,
    "SOFR-1B": [
        qfDiscountFactor(model, "SOFR-1B", date, payment_currency)
        for date in query_dates
    ],
    "Funding component only": [
        funding_component.discount_factor(Date(date))
        for date in query_dates
    ],
    "SOFR-1B-FLAT": [
        qfDiscountFactor(model, "SOFR-1B-FLAT", date, payment_currency)
        for date in query_dates
    ],
})

display(discount_factors.round(8))

,Tenor,Date,SOFR-1B,Funding component only,SOFR-1B-FLAT
0,Value date,2026-10-07,1.000000,1.0,1.000000
1,6M,2027-04-07,0.985152,1.0,0.985152
2,1Y,2027-10-07,0.970446,1.0,0.970446
3,2Y,2028-10-10,0.940565,1.0,0.940565
4,5Y,2031-10-07,0.852156,1.0,0.852156
5,10Y,2036-10-07,0.708223,1.0,0.708223
6,15Y,2041-10-07,0.582765,1.0,0.582765


## 5. Cashflow price and analytic risk

### 5.1 Create product

Create a `ProductFixedAccruedCashflow` using the inputs below.

**Hint:** Use `qfCreateProductFixedAccruedCashflow`; set the payment date to the termination date.

In [7]:
from fixedincomelib import qfCreateProductFixedAccruedCashflow

# Product inputs: edit these and rerun all three cells.
product_tenor = "5Y"
product_notional = 100.0
product_accrual_basis = "ACT/360"
termination_date = qfAddPeriod(value_date, product_tenor, "F", "USGS")

product = qfCreateProductFixedAccruedCashflow(
    effective_date=value_date,
    termination_date=termination_date,
    currency=payment_currency,
    notional=product_notional,
    accrual_basis=product_accrual_basis,
    payment_date=termination_date,
)


### 5.2 Price

Discount the product's payment using `SOFR-1B-FLAT`.

**Hint:** `product.accrued` is a year fraction. The fixed payment is $c=\text{product.notional}\times\text{product.accrued}$, so $V=c\,D_{\mathrm{SOFR-1B-FLAT}}(T)$.

In [8]:
c = product.notional * product.accrued
df = qfDiscountFactor(
    model, funding_bm.target, product.payment_date.ISO(), payment_currency
)
V = c * df

price_table = pd.DataFrame([{
    "Payment date": product.payment_date.ISO(),
    "Cashflow (c)": c,
    "Discount factor": df,
    "Price (V)": V,
}])
display(price_table.round(8))


,Payment date,Cashflow (c),Discount factor,Price (V)
0,2031-10-07,507.222222,0.852156,432.232456


### 5.3 Analytic risk

Compute $\nabla_{X^I}V$ for both components' IFR node values, holding node times and $c$ fixed. Use analytic derivatives, without bumping. Report derivatives per unit decimal IFR, without 1 bp scaling.

**Hint:** Use ACT/ACT (ISDA) for curve time $T$. The interpolator's `gradient_of_integrated_value_wrt_ordinate(0.0, T)` gives $w_i=\partial(\int_0^T f(s)\,ds)/\partial f_i$. Apply the chain rule to the full price:

$$\frac{\partial V}{\partial f_i}=-V\,w_i.$$

Include the funding node even when its IFR is zero; its derivative is $-TV$ for this single-node flat component.

In [9]:
# Curve time uses the model's day count, not the product's accrual basis.
T = qfAccrued(
    model.value_date.ISO(), product.payment_date.ISO(), "ACT/ACT (ISDA)"
)

projection_interpolator = projection_component.state_data_interpolator
funding_interpolator = funding_component.state_data_interpolator
projection_weights = projection_interpolator.gradient_of_integrated_value_wrt_ordinate(0.0, T)
funding_weights = funding_interpolator.gradient_of_integrated_value_wrt_ordinate(0.0, T)

# Chain rule: V already includes both SOFR and funding discount factors.
# Derivatives are per unit decimal IFR, with node times and c held fixed.
projection_risk = -V * projection_weights
funding_risk = -V * funding_weights

# Parameter order: SOFR nodes, then funding nodes.
X_I = np.concatenate([
    projection_component.state_data[1], funding_component.state_data[1]
])
risk = np.concatenate([projection_risk, funding_risk])

risk_table = pd.DataFrame({
    "Component": (
        [projection_bm.target] * len(projection_risk)
        + [funding_bm.target] * len(funding_risk)
    ),
    "Tenor": projection_data.axis1 + funding_data.axis1,
    "IFR (X_I)": X_I,
    "dV/dX_I": risk,
})
display(risk_table.round(8))


,Component,Tenor,IFR (X_I),dV/dX_I
0,SOFR-1B,1Y,0.030,-432.232456
1,SOFR-1B,2Y,0.031,-436.053599
2,SOFR-1B,3Y,0.032,-430.779710
3,SOFR-1B,4Y,0.033,-429.864059
4,SOFR-1B,5Y,0.034,-432.232456
5,SOFR-1B,6Y,0.035,-0.000000
6,SOFR-1B,7Y,0.036,-0.000000
7,SOFR-1B,8Y,0.037,-0.000000
8,SOFR-1B,9Y,0.038,-0.000000
9,SOFR-1B,10Y,0.039,-0.000000


## 6. Analytic checks and interpretation

For left-continuous piecewise-constant IFRs, the ordinate $f_i$ applies on $(x_{i-1},x_i]$. The first ordinate extends to the left and the last extends to the right. The integral weights $w_i$ are the lengths of the overlaps of those intervals with $[0,T]$; hence $\int_0^T f(s)\,ds=\sum_i f_iw_i$ and $\sum_iw_i=T$ for each component.

The full price is $V=c\exp[-\sum_i f_iw_i-\sum_j s_ju_j]$. Therefore $\partial V/\partial f_i=-Vw_i$ and $\partial V/\partial s_j=-Vu_j$. The funding curve has one flat node, so $u_1=T$ even when its IFR is zero. Nodes strictly after the payment interval have zero exposure. Product accrual uses ACT/360, while both curve components use ACT/ACT (ISDA).

The checks below use exact identities and independent overlap lengths; no input rate is bumped to calculate the submitted risk. The separate test suite also checks finite-difference agreement as an independent verification.


In [10]:
# Independent overlap lengths on [0, T] for nonnegative curve times.
def expected_ifr_weights(times, horizon):
    times = np.asarray(times, dtype=float)
    lower = np.r_[0.0, times[:-1]]
    upper = np.r_[times[:-1], np.inf]
    return np.maximum(0.0, np.minimum(horizon, upper) - lower)

expected_projection_weights = expected_ifr_weights(projection_component.state_data[0], T)
expected_funding_weights = expected_ifr_weights(funding_component.state_data[0], T)
np.testing.assert_allclose(projection_weights, expected_projection_weights, rtol=0, atol=1e-13)
np.testing.assert_allclose(funding_weights, expected_funding_weights, rtol=0, atol=1e-13)
np.testing.assert_allclose([projection_weights.sum(), funding_weights.sum()], [T, T], rtol=0, atol=1e-13)

integrated_sofr = np.dot(projection_component.state_data[1], expected_projection_weights)
integrated_spread = np.dot(funding_component.state_data[1], expected_funding_weights)
np.testing.assert_allclose(df, np.exp(-integrated_sofr - integrated_spread), rtol=1e-13)
np.testing.assert_allclose(risk, -V * np.r_[expected_projection_weights, expected_funding_weights], rtol=1e-13)
np.testing.assert_allclose(qfDiscountFactor(model, funding_bm.target, value_date), 1.0, atol=0)

if len(funding_component.state_data[1]) == 1:
    np.testing.assert_allclose(funding_risk[0], -T * V, rtol=1e-13)
if np.all(funding_component.state_data[1] == 0.0):
    np.testing.assert_allclose(discount_factors["Funding component only"], 1.0, atol=0)
    np.testing.assert_allclose(discount_factors["SOFR-1B"], discount_factors["SOFR-1B-FLAT"], rtol=0, atol=0)

print(f"ACT/360 product accrual: {product.accrued:.12f}")
print(f"ACT/ACT (ISDA) curve time: {T:.12f}")
print(f"Integrated SOFR IFR: {integrated_sofr:.12f}")
print(f"Integrated funding spread: {integrated_spread:.12f}")
print(f"Cashflow: {c:.12f}; discount factor: {df:.12f}; price: {V:.12f}")
print(f"Funding-node derivative per unit decimal IFR: {funding_risk[0]:.12f}")
print("All analytic notebook checks passed.")


ACT/360 product accrual: 5.072222222222
ACT/ACT (ISDA) curve time: 5.000000000000
Integrated SOFR IFR: 0.159985680066
Integrated funding spread: 0.000000000000
Cashflow: 507.222222222222; discount factor: 0.852155991697; price: 432.232455788284
Funding-node derivative per unit decimal IFR: -2161.162278941419
All analytic notebook checks passed.
